## Part 1: Train a Small BPE Tokenizer from Scratch

This script trains a BPE tokenizer on your own paragraph and prints the vocabulary and merge rules.



In [3]:
!pip install tokenizers


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import json

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

corpus = [
    "The unbelievably quick brown fox jumps over the lazy dog. "
    "Tokenization is a fundamental preprocessing step in natural language processing. "
    "Unbelievable transformations happen when neural networks learn representations. "
    "The industrialization of artificial intelligence continues to accelerate rapidly. "
    "Researchers study subword tokenization to handle rare and unknown words effectively."
]

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=300,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    min_frequency=1,
    show_progress=True,
)

tokenizer.train_from_iterator(corpus, trainer=trainer)

# --- Vocabulary ---
print("=" * 60)
print("VOCABULARY")
print("=" * 60)
vocab = tokenizer.get_vocab()
for token, idx in sorted(vocab.items(), key=lambda x: x[1]):
    print(f"{idx:4d}: {repr(token)}")

# --- Merge rules (from serialized model) ---
print("\n" + "=" * 60)
print("MERGE RULES")
print("=" * 60)

import json

tj = json.loads(tokenizer.to_str())
raw_merges = tj["model"]["merges"]

# Normalize: support both ["a b", ...] and [["a","b"], ...]
merges = []
for m in raw_merges:
    if isinstance(m, str):
        merges.append(tuple(m.split(" ")))
    else:
        merges.append(tuple(m))

for i, (a, b) in enumerate(merges[:50]):
    print(f"{i+1:3d}: {a} + {b} -> {a + b}")

print(f"\nTotal merges: {len(merges)}")

VOCABULARY
   0: '[UNK]'
   1: '[CLS]'
   2: '[SEP]'
   3: '[PAD]'
   4: '[MASK]'
   5: '.'
   6: 'R'
   7: 'T'
   8: 'U'
   9: 'a'
  10: 'b'
  11: 'c'
  12: 'd'
  13: 'e'
  14: 'f'
  15: 'g'
  16: 'h'
  17: 'i'
  18: 'j'
  19: 'k'
  20: 'l'
  21: 'm'
  22: 'n'
  23: 'o'
  24: 'p'
  25: 'q'
  26: 'r'
  27: 's'
  28: 't'
  29: 'u'
  30: 'v'
  31: 'w'
  32: 'x'
  33: 'y'
  34: 'z'
  35: 'ti'
  36: 'en'
  37: 'on'
  38: 'ra'
  39: 'ati'
  40: 'el'
  41: 'es'
  42: 'in'
  43: 'ation'
  44: 'he'
  45: 'or'
  46: 'pr'
  47: 'al'
  48: 'an'
  49: 'ar'
  50: 'iz'
  51: 'le'
  52: 'st'
  53: 'to'
  54: 'un'
  55: 'wor'
  56: 'ization'
  57: 'The'
  58: 'ab'
  59: 'bel'
  60: 'ces'
  61: 'ev'
  62: 'epr'
  63: 'ic'
  64: 'ial'
  65: 'iev'
  66: 'ken'
  67: 'ly'
  68: 'ne'
  69: 'ow'
  70: 'oces'
  71: 'sin'
  72: 'ura'
  73: 'ent'
  74: 'ations'
  75: 'and'
  76: 'word'
  77: 'believ'
  78: 'kenization'
  79: 'own'
  80: 'ocessin'
  81: 'ural'
  82: 'believab'
  83: 'ocessing'
  84: 'Res'
  85: 

## Part 2: Tokenize "unbelievably" Manually


In [11]:
# Tokenize the word "unbelievably"
word = "unbelievably"
encoding = tokenizer.encode(word)

print("=" * 60)
print(f"TOKENIZING: '{word}'")
print("=" * 60)
print(f"Token IDs: {encoding.ids}")
print(f"Tokens:    {encoding.tokens}")
print(f"Number of subword pieces: {len(encoding.tokens)}")

TOKENIZING: 'unbelievably'
Token IDs: [211]
Tokens:    ['unbelievably']
Number of subword pieces: 1


## Part 3: Compare tiktoken (GPT) vs. BERT WordPiece


In [12]:
import tiktoken
from transformers import BertTokenizer

sentence = "The unbelievably quick brown fox jumps over the lazy dog."

# --- GPT Tokenizer (tiktoken) ---
# Use cl100k_base (GPT-4) or o200k_base (GPT-4o, newer)
gpt_encoding = tiktoken.get_encoding("cl100k_base")
gpt_tokens = gpt_encoding.encode(sentence)

print("=" * 60)
print("GPT (tiktoken / cl100k_base)")
print("=" * 60)
print(f"Token count: {len(gpt_tokens)}")
print(f"Tokens: {[gpt_encoding.decode([t]) for t in gpt_tokens]}")

# --- BERT WordPiece Tokenizer ---
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_tokens = bert_tokenizer.tokenize(sentence)
bert_ids = bert_tokenizer.encode(sentence)

print("\n" + "=" * 60)
print("BERT (WordPiece / bert-base-uncased)")
print("=" * 60)
print(f"Token count (excluding special tokens): {len(bert_tokens)}")
print(f"Token count (including [CLS] and [SEP]): {len(bert_ids)}")
print(f"Tokens: {bert_tokens}")

# --- Comparison ---
print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"GPT tokens:  {len(gpt_tokens)}")
print(f"BERT tokens: {len(bert_tokens)}")
more_efficient = "GPT" if len(gpt_tokens) < len(bert_tokens) else "BERT"
print(f"More efficient (fewer tokens): {more_efficient}")

c:\SLM_Inference_Engine\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPT (tiktoken / cl100k_base)
Token count: 12
Tokens: ['The', ' unbelie', 'vably', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.']


c:\SLM_Inference_Engine\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gj261\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



BERT (WordPiece / bert-base-uncased)
Token count (excluding special tokens): 15
Token count (including [CLS] and [SEP]): 17
Tokens: ['the', 'un', '##bel', '##ie', '##va', '##bly', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']

COMPARISON
GPT tokens:  12
BERT tokens: 15
More efficient (fewer tokens): GPT


## Part 4: Find a Word That Splits Differently


In [13]:
test_words = [
    "photosynthesis",
    "antidisestablishmentarianism",
    "transformerification",  # made-up
    "quantumcomputing",
    "neuralnetwork",
    "blockchainification",   # made-up
    "tokenization",
    "hyperparameterization", # rare technical
]

print("=" * 80)
print(f"{'Word':<30} {'GPT Tokens':<25} {'BERT Tokens':<25}")
print("=" * 80)

for word in test_words:
    gpt_pieces = [gpt_encoding.decode([t]) for t in gpt_encoding.encode(word)]
    bert_pieces = bert_tokenizer.tokenize(word)
    
    gpt_str = str(gpt_pieces)
    bert_str = str(bert_pieces)
    
    print(f"{word:<30} {gpt_str:<25} {bert_str:<25}")

Word                           GPT Tokens                BERT Tokens              
photosynthesis                 ['photos', 'ynthesis']    ['photos', '##yn', '##thesis']
antidisestablishmentarianism   ['ant', 'idis', 'establish', 'ment', 'arian', 'ism'] ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']
transformerification           ['transform', 'er', 'ification'] ['transform', '##eri', '##fication']
quantumcomputing               ['quant', 'um', 'comput', 'ing'] ['quantum', '##com', '##put', '##ing']
neuralnetwork                  ['ne', 'ural', 'network'] ['neural', '##net', '##work']
blockchainification            ['block', 'chain', 'ification'] ['block', '##chai', '##ni', '##fication']
tokenization                   ['token', 'ization']      ['token', '##ization']   
hyperparameterization          ['hyper', 'parameter', 'ization'] ['hyper', '##para', '##meter', '##ization']
